In [ ]:
# Local environment setup (converted from Kaggle)
import numpy as np
import pandas as pd
import os
import torch

# Create working directories for processed data
os.makedirs("./working/train", exist_ok=True)
os.makedirs("./working/test", exist_ok=True)
os.makedirs("./working/val_task3", exist_ok=True)

# GPU setup
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    torch.backends.cudnn.benchmark = True

# Subtask 01 - Data Exploration

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri

## 1. Loading Data

In [ ]:
def load_data(data_path):
    data = np.load(data_path, allow_pickle=True).item()
    return data

data_path = "./elec-70127-fno/data/train/data_02005.npy"

result = load_data(data_path)

for key, item in result.items():
    print(key, item.shape)

## 2. Data Visulisation

In [ ]:
def visulise_data(data_path, cmap="jet"):
    zeta = 10

    data = load_data(data_path)

    coord = data.get("coord")
    face = data.get("cell_node_list")
    value = data.get("field_node")[:, 2] # here we are visulising the pressure field

    left, right = min(coord[:, 0]), max(coord[:, 0])
    bottom, top = min(coord[:, 1]), max(coord[:, 1])

    width = zeta * (right - left)
    height = zeta * (top - bottom)

    triangle = tri.Triangulation(coord[:, 0], coord[:, 1], face)

    fig, ax = plt.subplots(figsize=(width, height))

    ax.set_aspect("equal")
    ax.triplot(triangle, 'ko-', ms=0.1, lw=0.1)
    ax.tripcolor(triangle, value, cmap=cmap, alpha=0.4)

    return fig, ax

In [ ]:
visulise_data(data_path)

## 3. Data statistics

In [ ]:
import glob
from natsort import natsorted

In [ ]:
# Make it a task
def get_mean_std(data_folder_path):
    data_list = []

    file_path = os.path.join(data_folder_path, "data_*.npy")
    file_names = glob.glob(file_path)
    file_names = natsorted(file_names)

    for file_name in file_names:
        data = load_data(file_name)
        data_list.append(data.get("field_node"))

    data = np.concatenate(data_list, axis=0)

    mean = np.mean(data, axis=0, keepdims=True)
    std = np.std(data, axis=0, keepdims=True)

    return mean, std


In [ ]:
mean, std = get_mean_std("./elec-70127-fno/data/train/")

In [ ]:
from scipy.interpolate import griddata, LinearNDInterpolator
from scipy.spatial import Delaunay


def get_grid_and_mask(data, tri_cache=None):
    """
    Interpolate unstructured mesh data onto a regular grid.
    
    Optimisation: if tri_cache is provided (a precomputed Delaunay triangulation),
    reuse it instead of rebuilding for every file (~3-5x speedup).
    """
    r = 0.05
    x_center = 0.3
    y_center = 0.2

    grid_num_x = 320
    grid_num_y = 80

    coord = data.get("coord")
    field_node = data.get("field_node")

    x_min, x_max = min(coord[:, 0]), max(coord[:, 0])
    y_min, y_max = min(coord[:, 1]), max(coord[:, 1])

    x_coord, y_coord = np.meshgrid(
        np.linspace(x_min, x_max, grid_num_x),
        np.linspace(y_min, y_max, grid_num_y)
    )

    x_coord, y_coord = x_coord.flatten(), y_coord.flatten()

    # calculate mask
    dis_from_center = np.sqrt((x_coord - x_center)**2 + (y_coord - y_center)**2)
    mask = dis_from_center <= r

    # Build Delaunay triangulation once, reuse across files
    if tri_cache is None:
        tri_cache = Delaunay(coord)

    grid_data = []
    xi = np.column_stack([x_coord, y_coord])
    num_dim = field_node.shape[1]
    for i in range(num_dim):
        interp = LinearNDInterpolator(tri_cache, field_node[:, i])
        grid_data_i = interp(xi)

        grid_data_i[mask] = 0
        grid_data_i = np.nan_to_num(grid_data_i, nan=0.0)
        grid_data_i = grid_data_i.reshape(grid_num_y, grid_num_x)
        grid_data.append(grid_data_i)

    grid_data = np.stack(grid_data, axis=0)
    mask_grid = mask.reshape(1, grid_num_y, grid_num_x)

    return grid_data, mask_grid, tri_cache

In [ ]:
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import multiprocessing


def _process_one_file(args):
    """Worker function for parallel post-processing."""
    file_name, save_dir = args
    data = np.load(file_name, allow_pickle=True).item()
    grid_data, mask, _ = get_grid_and_mask(data)
    data["field_conv"] = grid_data
    data["mask"] = mask

    if save_dir is None:
        np.save(file_name, data)
    else:
        base_name = os.path.basename(file_name)
        np.save(os.path.join(save_dir, base_name), data)
    return True


def post_process(dataset_dir, save_dir=None, n_workers=None):
    """
    Parallel post-processing: interpolate mesh -> grid for all .npy files.
    
    - Skips if output directory already has the expected number of files.
    - Uses ThreadPoolExecutor (works in Jupyter notebooks on Windows).
    - scipy's LinearNDInterpolator releases the GIL in C code, so threads
      provide real parallelism for this workload.
    """
    file_names = glob.glob(os.path.join(dataset_dir, "data_*.npy"))
    file_names = natsorted(file_names)

    # Skip if already processed
    if save_dir is not None:
        existing = glob.glob(os.path.join(save_dir, "data_*.npy"))
        if len(existing) >= len(file_names):
            print(f"Skipping {save_dir} â€” already processed ({len(existing)} files)")
            return

    os.makedirs(save_dir, exist_ok=True) if save_dir else None

    if n_workers is None:
        n_workers = min(multiprocessing.cpu_count(), 8)

    args_list = [(f, save_dir) for f in file_names]

    print(f"Processing {len(file_names)} files with {n_workers} threads...")
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        list(tqdm(executor.map(_process_one_file, args_list),
                  total=len(file_names), desc="Processing files"))

In [ ]:
post_process(dataset_dir="./elec-70127-fno/data/train/", save_dir="./working/train")

In [ ]:
post_process(dataset_dir="./elec-70127-fno/data/test/", save_dir="./working/test")

In [ ]:
post_process(dataset_dir="./elec-70127-fno/data/val_task3/", save_dir="./working/val_task3")

In [ ]:
data_root = "./working/"

In [ ]:
data = load_data(data_root + "train/data_02005.npy")

grid_data = data["field_conv"]
print(grid_data.shape)
plt.imshow(grid_data[0], cmap="jet")
plt.show()

# Subtask 02 - Dataset construction and Data Loading

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from natsort import natsorted
from scipy.interpolate import griddata

In [ ]:
import glob
class CNNDataset(Dataset):
    """
    A custom PyTorch dataset for loading spatiotemporal data in a format suitable for CNN-based models.

    This dataset handles time-series data stored in `.npy` files,
    The dataset loads consecutive time frames, applies transformations,
    and formats the data for training models.

    Args:
        dataset_dir (str, optional): Path to the directory containing `.npy` dataset files.
        t_dim (int, optional): The number of time steps to include per sample. Default is 10.
        transform (callable, optional): A function to apply transformations to input data.
        target_transform (callable, optional): A function to apply transformations to target data.

    Attributes:
        file_names (list): A sorted list of file paths to dataset files.
        t_dim (int): The number of time steps per sample.
        transform (callable): Transformation function for input data.
        target_transform (callable): Transformation function for target data.

    Task:
    - Implement `__len__`:
        - Compute the number of samples by dividing dataset length by `(2 * t_dim)`.
    - Implement `__getitem__`:
        - Compute index ranges for `in_data` and `out_data`.
        - Load and process the input-output data pairs.
        - Apply transformations if provided.
        - Stack time-series data along the temporal axis.
        - Ensure the final shape follows `(x1, x2, t, channel)` format. where x1 and x2 are spatial dimensions and t is the number of time steps.

    Note:
    - This dataset assumes `.npy` files contain dictionary-like data.
    - Set `t_dim == 10` based on the sequence length training.
    - Feel free to modify the preprocessing pipeline to fit specific model architectures - if that is necessary.
    """

    def __init__(self, dataset_dir=None, t_dim=10, transform=None, target_transform=None):
        self.t_dim=t_dim
        self.transform = transform
        self.target_transform = target_transform

        self.file_names = glob.glob(os.path.join(dataset_dir, "data_*.npy"))
        self.file_names = natsorted(self.file_names)

    def __len__(self):
        # Each sample requires t_dim input frames + t_dim output frames = 2*t_dim consecutive files
        return len(self.file_names) // (2 * self.t_dim)

    def load_data(self, data_path):
        data = np.load(data_path, allow_pickle=True).item()
        return data

    def __getitem__(self, idx):
        # Compute the start index for this sample's window
        start_idx = idx * 2 * self.t_dim

        # Load t_dim consecutive frames for input
        in_list = []
        for i in range(self.t_dim):
            data = self.load_data(self.file_names[start_idx + i])
            in_list.append(data.get("field_conv"))  # (3, 80, 320)

        # Load t_dim consecutive frames for output (target)
        out_list = []
        for i in range(self.t_dim):
            data = self.load_data(self.file_names[start_idx + self.t_dim + i])
            out_list.append(data.get("field_conv"))  # (3, 80, 320)

        # Stack along temporal axis: (3, 80, 320) x t_dim -> (3, 80, 320, t_dim)
        in_data = np.stack(in_list, axis=-1)   # (3, 80, 320, 10)
        out_data = np.stack(out_list, axis=-1)  # (3, 80, 320, 10)

        # Apply transforms (normalisation operates on channel-first format)
        if self.transform:
            in_data = self.transform(in_data)
        if self.target_transform:
            out_data = self.target_transform(out_data)

        # Permute to (x1, x2, t, channel) = (80, 320, 10, 3) as required by FNO3d
        if isinstance(in_data, torch.Tensor):
            in_data = in_data.permute(1, 2, 3, 0)
            out_data = out_data.permute(1, 2, 3, 0)
        else:
            in_data = np.transpose(in_data, (1, 2, 3, 0))
            out_data = np.transpose(out_data, (1, 2, 3, 0))

        return in_data, out_data

In [ ]:
dataset_dir = f"{data_root}/test/"
print(dataset_dir)

dataset = CNNDataset(dataset_dir)
print(len(dataset))

idx = 20
sample, target = dataset[idx]
print(sample.shape, target.shape)

## 2. Data Transformation

In [ ]:
class Normalise:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, data):
        data = (data - self.mean) / self.std
        return data


class ToFloatTransform:
    def __init__(self, scale=True):
        return

    def __call__(self, data):
        data = torch.from_numpy(data).float()
        return data


class Compose:
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, data):
        for transform in self.transforms:
            data = transform(data)
        return data

In [ ]:
transform = Compose([
    Normalise(mean = mean.reshape(-1, 1, 1, 1),
              std = std.reshape(-1, 1, 1, 1)),
    ToFloatTransform(),
])

dataset = CNNDataset(dataset_dir = dataset_dir, transform=transform, target_transform=transform)

idx = 0
data_in, target = dataset[idx]

conv_in = data_in
print(conv_in.shape)

## 3. Data Loader

In [ ]:
Loader = DataLoader(dataset=dataset, num_workers=0, batch_size=16, pin_memory=True)

# Subtask 03 - FNO

![](https://www.googleapis.com/download/storage/v1/b/kaggle-user-content/o/inbox%2F24644801%2Fee12956a1e03360c33eacf463cc67094%2FWX20260202-0053412x.png?generation=1769993637576752&alt=media)

## Understanding `SpectralConv3d` Class

1. In the `class SpectralConv3d(nn.Module)` class, why we use `self.weights1, self.weights2 ...`

2. Why there is a 'self.scale = (1 / (in_channels * out_channels))` in the code? what is the purpose of this line?

3. What is the line `torch.einsum("bixyz,ioxyz->boxyz", input, weights)` doing here? Could you explain it? Could you also give a pytorch implementation with out einsum doing the same thing (porvide demo in the code cell below)? Why we want to use einsum?

4. Try to go through the `forward` methods, annotate this methods line by line explaining the purpose of each line.

## Your Answers here:
1.
...

In [ ]:
# Tier 1 improvements: GELU activation (Best Practices in Operator Learning, 2024)

from torch import nn
import torch.nn.functional as F
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SpectralConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2, modes3):
        super(SpectralConv3d, self).__init__()
        """
        3D Fourier layer. It does FFT, linear transform, and Inverse FFT.
        """
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        self.modes3 = modes3

        self.scale = (1 / (in_channels * out_channels))
        self.weights1 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3, dtype=torch.cfloat))
        self.weights3 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3, dtype=torch.cfloat))
        self.weights4 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3, dtype=torch.cfloat))

    def compl_mul3d(self, input, weights):
        return torch.einsum("bixyz,ioxyz->boxyz", input, weights)

    def forward(self, x):
        batchsize = x.shape[0]
        x_ft = torch.fft.rfftn(x, dim=[-3,-2,-1])

        out_ft = torch.zeros(batchsize, self.out_channels, x.size(-3), x.size(-2), x.size(-1)//2 + 1, dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes1, :self.modes2, :self.modes3] = \
            self.compl_mul3d(x_ft[:, :, :self.modes1, :self.modes2, :self.modes3], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2, :self.modes3] = \
            self.compl_mul3d(x_ft[:, :, -self.modes1:, :self.modes2, :self.modes3], self.weights2)
        out_ft[:, :, :self.modes1, -self.modes2:, :self.modes3] = \
            self.compl_mul3d(x_ft[:, :, :self.modes1, -self.modes2:, :self.modes3], self.weights3)
        out_ft[:, :, -self.modes1:, -self.modes2:, :self.modes3] = \
            self.compl_mul3d(x_ft[:, :, -self.modes1:, -self.modes2:, :self.modes3], self.weights4)

        x = torch.fft.irfftn(out_ft, s=(x.size(-3), x.size(-2), x.size(-1)))
        return x



class FNO3d(nn.Module):
    def __init__(self,
                 modes1=8,
                 modes2=8,
                 modes3=8,
                 width=36
                ):
        super(FNO3d, self).__init__()
        """
        3D Fourier Neural Operator with GELU activation (Tier 1 improvement).
        GELU is smoother than ReLU and avoids dead neurons, consistently improving
        neural operator performance (Enyeart et al., "Best Practices in Operator Learning", 2024).
        """
        self.modes1 = modes1
        self.modes2 = modes2
        self.modes3 = modes3
        self.width = width

        self.fc0 = nn.Linear(3, self.width)
        self.conv0 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)
        self.conv1 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)
        self.conv2 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)

        self.w0 = nn.Conv1d(self.width, self.width, 1)
        self.w1 = nn.Conv1d(self.width, self.width, 1)
        self.w2 = nn.Conv1d(self.width, self.width, 1)

        self.fc1 = nn.Linear(self.width, 128)
        self.fc2 = nn.Linear(128, 3)

    def forward(self, x):
        batchsize = x.shape[0]
        size_x, size_y, size_z = x.shape[1], x.shape[2], x.shape[3]

        x = self.fc0(x)
        x = x.permute(0, 4, 1, 2, 3)

        # Fourier Layer 0
        x1 = self.conv0(x)
        x2 = self.w0(x.view(batchsize, self.width, -1))
        x2 = x2.view(batchsize, self.width, size_x, size_y, size_z)
        x = x1 + x2
        x = F.gelu(x)  # GELU instead of ReLU

        # Fourier Layer 1
        x1 = self.conv1(x)
        x2 = self.w1(x.view(batchsize, self.width, -1))
        x2 = x2.view(batchsize, self.width, size_x, size_y, size_z)
        x = x1 + x2
        x = F.gelu(x)

        # Fourier Layer 2
        x1 = self.conv2(x)
        x2 = self.w2(x.view(batchsize, self.width, -1))
        x2 = x2.view(batchsize, self.width, size_x, size_y, size_z)
        x = x1 + x2
        x = F.gelu(x)

        x = x.permute(0, 2, 3, 4, 1)
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)

        return x

In [ ]:
model = FNO3d(10, 10, 5)
dummy = torch.randn((8, 90, 320, 10, 3))
print(model(dummy).shape)

In [ ]:
# =============================================================================
# Tier 1 Training: RelativeL2 loss, noise injection, tqdm, best model saving
# =============================================================================

import matplotlib.pyplot as plt
from tqdm import tqdm


def get_run_dir(version="v1"):
    """Create a new numbered run directory: ./runs/vX/run_NNN/"""
    base = f"./runs/{version}"
    os.makedirs(base, exist_ok=True)
    existing = [d for d in os.listdir(base) if d.startswith("run_")]
    run_num = len(existing) + 1
    run_dir = os.path.join(base, f"run_{run_num:03d}")
    os.makedirs(run_dir, exist_ok=True)
    print(f"Saving to: {run_dir}")
    return run_dir


def get_latest_run_dir(version="v1"):
    """Find the latest run directory for a given version."""
    base = f"./runs/{version}"
    if not os.path.exists(base):
        return "."
    existing = sorted([d for d in os.listdir(base) if d.startswith("run_")])
    if not existing:
        return "."
    return os.path.join(base, existing[-1])


class RelativeL2Loss(torch.nn.Module):
    """Relative L2 norm loss: ||pred - target||_2 / ||target||_2"""
    def forward(self, pred, target):
        diff_norm = torch.norm(pred - target)
        target_norm = torch.norm(target)
        return diff_norm / (target_norm + 1e-8)


def train_one_epoch(model, loader, optimizer, criterion, noise_std=0.01):
    model.train()
    avg_loss = 0.0
    for batch in tqdm(loader, desc="Training", leave=False):
        in_data, out_data = batch
        in_data, out_data = in_data.to(device, non_blocking=True), out_data.to(device, non_blocking=True)
        in_data = in_data + noise_std * torch.randn_like(in_data)
        optimizer.zero_grad()
        pred = model(in_data)
        loss = criterion(pred, out_data)
        loss.backward()
        optimizer.step()
        avg_loss += loss.item()
    avg_loss /= len(loader)
    return avg_loss, model


def test_one_epoch(model, loader, criterion):
    model.eval()
    avg_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(loader, desc="Testing", leave=False):
            in_data, out_data = batch
            in_data, out_data = in_data.to(device, non_blocking=True), out_data.to(device, non_blocking=True)
            pred = model(in_data)
            loss = criterion(pred, out_data)
            avg_loss += loss.item()
    avg_loss /= len(loader)
    return avg_loss


def plot_losses(train_losses, test_losses, test_epochs, run_dir):
    """Plot train and test losses and save to run directory."""
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(range(1, len(train_losses) + 1), train_losses, label="Train Loss", alpha=0.8)
    if test_losses:
        ax.plot(test_epochs, test_losses, 'o-', label="Test Loss", alpha=0.8)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training & Test Loss (v1 - Tier 1)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_yscale("log")
    fig.tight_layout()
    fig.savefig(os.path.join(run_dir, "loss_plot.png"), dpi=150)
    plt.show()
    plt.close(fig)


def train(epoch, model, optimizer, loader_train, loader_test, scheduler=None, version="v1", save_prefix="best_model"):
    criterion = RelativeL2Loss()
    model = model.to(device)
    run_dir = get_run_dir(version)
    best_test_loss = float("inf")
    train_losses = []
    test_losses = []
    test_epochs = []
    for i in tqdm(range(epoch), desc="Epochs"):
        loss, model = train_one_epoch(
            model=model, loader=loader_train, optimizer=optimizer, criterion=criterion)
        train_losses.append(loss)
        if scheduler is not None:
            scheduler.step()
        lr_current = optimizer.param_groups[0]['lr']
        print(f"Epoch \t{i+1}, Train Loss \t{loss:.6f}, LR \t{lr_current:.6f}")
        if (i+1) % 5 == 0:
            info = test_one_epoch(
                model=model, loader=loader_test, criterion=criterion)
            test_losses.append(info)
            test_epochs.append(i + 1)
            print(f"\t Epoch \t{i+1}, Test Loss \t{info:.6f}")
            if info < best_test_loss:
                best_test_loss = info
                torch.save(model.state_dict(), os.path.join(run_dir, f"{save_prefix}.pth"))
                print(f"\t >> New best model saved (test loss: {info:.6f})")
            torch.save(model.state_dict(), os.path.join(run_dir, f"epoch_{i+1}.pth"))
    # Save loss plot
    plot_losses(train_losses, test_losses, test_epochs, run_dir)
    print(f"Training complete. Best test loss: {best_test_loss:.6f}, saved in {run_dir}")
    return run_dir

## Model Training and Tuning

Key Considerations:

What is the role of modes1, modes2, and modes3?
> These parameters define the number of Fourier modes retained in each spatial dimension, controlling the spectral resolution of the model.

What loss function did you use to train the model?
> Specify the loss function(s) experimented with and justify the choice in terms of stability, convergence, and accuracy.

Model Tuning and Optimization Log:

Provide a concise summary of your model tuning process, including:
* Alterations made to the baseline model to improve performance.
* Hyperparameter adjustments and their impact.
* The most effective modification that led to the best overall results.

Clearly document your reasoning behind each change to demonstrate an iterative, data-driven approach to model optimization.

In [ ]:
# Tier 1 hyperparameters (SOTA-informed) + GPU-optimized DataLoaders
epoch = 200
lr = 1e-3
bs = 4

model = FNO3d(modes1=16, modes2=10, modes3=5, width=48)

train_set = CNNDataset(
    dataset_dir=f"{data_root}/train/",
    transform=transform, target_transform=transform)
test_set = CNNDataset(
    dataset_dir=f"{data_root}/test/",
    transform=transform, target_transform=transform)

loader_train = DataLoader(train_set, batch_size=bs, num_workers=0, pin_memory=True)
loader_test = DataLoader(test_set, batch_size=bs, num_workers=0, pin_memory=True)

optimizer = torch.optim.Adam(
    model.parameters(), lr=lr, weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=epoch, eta_min=1e-6
)

current_run_dir = train(epoch=epoch, model=model, optimizer=optimizer,
      loader_train=loader_train, loader_test=loader_test, scheduler=scheduler)

In [ ]:
def load_model(model, weight_path, strict=False):
    """
    Loads pre-trained weights into a PyTorch model from a given file path.

    Args:
        model (torch.nn.Module): The PyTorch model.
        weight_path (str): File path to the pre-trained model weights.

    Returns:
        torch.nn.Module: The model with loaded weights.
    """
    try:
        model.load_state_dict(torch.load(weight_path), strict=strict)
    except RuntimeError:
        state_dict = torch.load(weight_path, map_location=device)
        model.load_state_dict(state_dict, strict=strict)
    return model


def denormalise(data, mean, std):
    return data * std + mean

def normalise(data, mean, std):
    return (data - mean) / std

def calc_norm_error(pred, truth, ord=2):
    """
    Compute the relative error of two input. input are supposed to have
    shape of (channel, height, width). Other shape might work but please
    be careful and aware of what you are doing.

    pred: The prediction output by model.
    truth: The ground truth against which error are measured.
    ord: the order of norm.
    """
    pred, truth = [vec.flatten() for vec in [pred, truth]]
    nume = torch.linalg.norm((pred - truth), ord=ord)
    deno = torch.linalg.norm(truth, ord=ord)
    return nume / deno

def calculate_error_and_inference(dataloader, model, mean, std):
    model.to(device)
    model.eval()
    error = 0.0
    pred_list = []
    ground_list = []
    with torch.no_grad():
        for batch in dataloader:
            data_in, target = batch
            data_in, target = data_in, target
            data_in, target = data_in.to(device), target.to(device)
            pred = model(data_in)
            pred, target = denormalise(pred, mean, std), denormalise(target, mean, std)
            pred_list.append(pred.detach().cpu().numpy())
            ground_list.append(target.detach().cpu().numpy())
            error_i = calc_norm_error(pred, target)
            error += error_i.detach()
    error /= (len(dataloader) * dataloader.batch_size)
    return error, pred_list, ground_list


In [ ]:
# Evaluate Tier 1 model (best on test set)
model = FNO3d(modes1=16, modes2=10, modes3=5, width=48)

run_dir = get_latest_run_dir("v1")
weight_path = os.path.join(run_dir, "best_model.pth")
if not os.path.exists(weight_path):
    weight_path = "./best_model.pth" if os.path.exists("./best_model.pth") else "./epoch_200.pth"
model = load_model(model, weight_path)
print(f"Loaded weights from: {weight_path}")

dataset_test = CNNDataset(
    dataset_dir=f"{data_root}/test/",
    transform=transform, target_transform=transform)
test_loader = DataLoader(dataset_test, batch_size=1, num_workers=0, pin_memory=True)

error, pred, ground = calculate_error_and_inference(
    test_loader, model,
    mean=torch.tensor(mean.reshape(1, 1, 1, 1, -1)).to(device),
    std=torch.tensor(std.reshape(1, 1, 1, 1, -1)).to(device),
)
print(f"Relative L2 Error: {error:.6f}")

In [ ]:
def pred_to_df(data):
    reshaped = data.reshape(-1, 3)
    df = pd.DataFrame(reshaped, columns=['Feature_1', 'Feature_2', 'Feature_3'])
    df.index.name = "ROW_ID"
    return df

In [ ]:
file_dir = f"{data_root}/val_task3/"
file_names = glob.glob(os.path.join(file_dir, "data_*.npy"))
file_names = natsorted(file_names)

in_list = []
for i in range(len(file_names)):
    in_data = load_data(file_names[i]).get("field_conv")
    in_data = normalise(
        in_data,
        mean = mean.reshape(-1, 1, 1),
        std = std.reshape(-1, 1, 1),
    )
    in_list.append(in_data)

in_data = np.stack(in_list, axis=0)
in_data = in_data.transpose(2, 3, 0, 1)
in_data = torch.tensor(in_data).unsqueeze(0).float().to(device)
print(in_data.shape)

model.to(device)
model.eval()
with torch.no_grad():
    pred = model(in_data).detach().cpu().numpy()
    pred = denormalise(pred, mean, std)
print(pred.shape)

In [ ]:
pred = np.stack(pred, axis=0)
print(pred.shape)

In [ ]:
import pandas as pd
print(pred.shape)
# save file for submission in run directory
pred_1 = np.stack(pred, axis=0)
run_dir = current_run_dir if 'current_run_dir' in dir() else get_latest_run_dir("v1")
np.save(os.path.join(run_dir, "pred_1.npy"), pred_1)
pred_to_df(pred_1).to_csv(os.path.join(run_dir, "submission.csv"), index=True)
print(f"Submission saved to: {os.path.join(run_dir, 'submission.csv')}")

In [ ]:
import matplotlib.tri as tri
cmap = "jet"
zeta = 10

coord = data["coord"]
face = data["cell_node_list"]

left, right = min(coord[:, 0]), max(coord[:, 0])
bottom, top = min(coord[:, 1]), max(coord[:, 1])
idx = 9

value = pred[0, :, :, idx, 0]
plt.imshow(value)

## Generative AI Usage Log

Document your usage of generative AI.

### Case 1:

- **Tool Used:**  

- **Question / Context:**  

- **Prompt / Process:**  

- **How You Used the Output:**

### Case 2:

...


